# Agent identity & access governance demo with fakes (Phase IDN)

> Notebook generated from [`examples/agent_identity.py`](https://github.com/prismal-ai/prismal/blob/main/examples/agent_identity.py).

| Field | Value |
|------|-------|
| **API key** | 🔑 Not required — runs offline with injected fakes |
| **Source** | `examples/agent_identity.py` |

Drives the identity layer without any LLM, IdP, or network:

1. Issue a verifiable per-agent identity (a `did:key`) with least-privilege
   scopes from a `LocalIdentityProvider`.
2. Store and resolve a scoped secret through a `FakeVault` — only a
   `SecretStr` ever leaves the vault, and an out-of-scope resolve is rejected.
3. Authorize tool calls through an identity `PolicyEngine`: in-scope is
   allowed, out-of-scope is denied, a destructive op routes to HITL.
4. Mint and propagate an OAuth on-behalf-of token (scopes only narrow).


In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports

In [ ]:
from __future__ import annotations

from pydantic import SecretStr

from prismal.core.exceptions import ScopeError
from prismal.identity import (
    EnvVault,
    FakeVault,
    IdentityPolicy,
    LocalIdentityProvider,
    PolicyEffect,
    PolicyEngine,
    Scope,
    mint_on_behalf,
    propagate,
    validate,
)

## The demo

In [ ]:
def main() -> None:
    # 1. Issue a verifiable identity for the `coder` agent, scoped to file writes.
    provider = LocalIdentityProvider()
    coder = provider.issue(agent_name="coder", scopes=(Scope("tools:write_file"),))
    print(f"issued: {coder.did}")
    print(f"verify: {provider.verify(coder.did)}  scopes={[s.resource for s in coder.scopes]}")

    # 2. Store + resolve a scoped secret. The raw value never prints.
    vault: FakeVault | EnvVault = FakeVault()
    vault.store("openai", SecretStr("sk-secret"), scopes=(Scope("net:egress"),))
    cred = vault.resolve("openai", scopes=(Scope("net:egress"),))
    print(f"resolved credential ref={cred.ref!r} value={cred.value!r}")  # masked
    try:
        vault.resolve("openai", scopes=(Scope("tools:write_file"),))
    except ScopeError as exc:
        print(f"out-of-scope resolve rejected: {exc}")

    # 3. Authorize tool calls through the policy engine.
    engine = PolicyEngine(
        [
            IdentityPolicy(
                "coder", "tool_call", "tools:write_file", PolicyEffect.ALLOW, "tools:write_file"
            ),
            IdentityPolicy("coder", "tool_call", "tools:delete_file", PolicyEffect.REQUIRE_HITL),
            IdentityPolicy("*", "tool_call", "*", PolicyEffect.DENY),  # default-deny catch-all
        ]
    )
    for resource in ("tools:write_file", "tools:delete_file", "secrets:db"):
        decision = engine.allow(identity=coder, action="tool_call", resource=resource)
        print(f"policy {resource:<20} -> {decision.effect.value} ({decision.rule})")

    # 4. On-behalf-of delegation — scopes can only narrow along the chain.
    token = mint_on_behalf(
        subject="user@example.com",
        scopes=(Scope("rag:*"),),
        ttl_s=900,
        issuer=coder.did,
    )
    narrowed = propagate(token, via="did:key:zSubAgent", scopes=(Scope("rag:read"),))
    print(f"on-behalf chain: {narrowed.chain}")
    print(f"validate rag:read -> {validate(narrowed, action='tool_call', resource='rag:read')}")
    print(f"validate rag:write -> {validate(narrowed, action='tool_call', resource='rag:write')}")

---

## Run the demo

The original file ends with a plain `main()` call — `main` is synchronous
here, so no `await` is needed:

```python
main()
```

In [ ]:
# Uncomment to run the end-to-end demo (runs offline — no API key needed).
# main()